# Logistic Regression – Detailed Explanation
## 1. Overview
In this notebook, we are building a Logistic Regression model to solve a classification problem. The objective is to predict a binary outcome (e.g., 0 or 1, No or Yes) based on input features.

Logistic Regression is a supervised machine learning algorithm used primarily for classification problems. Despite its name, it is a classification model, not a regression model. It predicts the probability that a given input belongs to a particular class.
It is most commonly used for:

Binary classification (e.g., spam vs. not spam, churn vs. not churn)
Can also be extended to multiclass classification (e.g., using One-vs-Rest or Softmax)

2. Intuition
Linear regression predicts continuous values, but classification requires outputs bounded between 0 and 1.

Logistic Regression solves this by:

    a. Computing a linear combination of input features:
        z=wTx+bz = w^T x + bz=wTx+b


    b. Passing this value through the sigmoid (logistic) function:
        σ(z)=11+e−z\sigma(z) = \frac{1}{1 + e^{-z}}σ(z)=1+e−z1​

This converts any real number into a probability:

    Output close to 1 → Class 1
    Output close to 0 → Class 0



In [0]:
# Note that it would be neccessary to import the dataset as we did in training_model notebook 
# But for now we don't need to reimport the data set from the given url 


In [0]:
# Read the dataset using pyspark 
from pyspark.sql import types as T
from pyspark.sql import functions as F
data = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/mlpractice/source/mlmodel/diabetes_data/*.csv")
train, test = data.randomSplit([0.7, 0.3])

display(data.limit(5))

In [0]:
import time
import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score

def train_model(training_data, test_data, maxIterations, regularizationParam):
    
    # Start ML-flow run
    with mlflow.start_run():
        numFeaturs = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
        
        # Covert spark DataFrame to pandas DataFrame
        training_pd = training_data.toPandas()
        test_pd = test_data.toPandas()
        
        # Scale features
        scaler = MinMaxScaler()
        train_scaled = scaler.fit_transform(training_pd[numFeaturs])
        test_scaled = scaler.transform(test_pd[numFeaturs])


        #Train sklearn model: LogisticRegression
        lr = LogisticRegression(max_iter=maxIterations, C=1.0/regularizationParam, random_state=42)
        lr.fit(train_scaled, training_pd['Outcome'])
        
        # Evaluate the trained model: Make predictions
        y_pred = lr.predict(test_scaled)
        y_actual = test_pd['Outcome']

        accuracy = accuracy_score(y_actual, y_pred)
        recall = recall_score(y_actual, y_pred, average='weighted')
        precision = precision_score(y_actual, y_pred, average='weighted')
        
        print("accuracy: %s" % accuracy)
        print("weightedRecall: %s" % recall)
        print("weightedPrecision: %s" % precision)

        # Log the metrics 
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("weightedRecall", recall)
        mlflow.log_metric("weightedPrecision", precision)
        
        # Create Signature 
        input_example = pd.DataFrame([
            {'Pregnancies': 8, 'Glucose': 85, 'BloodPressure': 65, 'SkinThickness': 29, 'Insulin': 0, 'BMI': 26.6, 'DiabetesPedigreeFunction': 0.675, 'Age': 34}
        ])

        output_example = pd.DataFrame({'prediction': [1]})
        
        signature = infer_signature(input_example, output_example)
        
        # Log the model with signature 
        model_name = "diabetes_sklearn_"+str(int(time.time()))
        mlflow.sklearn.log_model(
            sk_model = lr, 
            name = model_name,
            signature=signature, 
            input_example = input_example
        )


        # Log the parameters
        mlflow.log_param("maxIter", maxIterations)
        mlflow.log_param("regParam", regularizationParam)

        print("Experiment run ID: {}".format(mlflow.active_run().info.run_id))
         
       
 
        

In [0]:
train_model(train,test, 5, 0.05)

In [0]:
train_model(train, test, 10, 0.2)